# ভিশনের বাইরে: পূর্ণ মাল্টিমোডালিটি

প্রকৃত multimodal model-এর তিনটি বৈশিষ্ট্য, প্রতিটিই scratch থেকে তৈরি ও পরিমাপ করা।

কোনো download নেই, কোনো pretrained weight নেই। কেবল CPU, কয়েক মিনিট।

1. **EMERGENT CROSS-MODAL ALIGNMENT (ImageBind-এর ফলাফল)।** (text, image) জোড়া আর (text, audio) জোড়া train করুন — এবং **কখনোই** একটিও (image, audio) জোড়া নয় — তারপর image->audio retrieval মাপুন। যেহেতু দুটি modality-কেই একই text anchor-এর দিকে টানা হয়েছিল, তারা বিনামূল্যেই একে অপরের সঙ্গে aligned হয়ে যায়। এই কারণেই `n` সংখ্যক modality-র জন্য একটি omni-model-এর O(n^2) paired dataset দরকার হয় না।
2. **GENERATION হলো NEXT-TOKEN PREDICTION (Chameleon-এর ফলাফল)।** ইমেজকে discrete token-এ quantize করুন, সেগুলোকে টেক্সটের **একই** vocabulary-তে রাখুন, এবং একটি objective দিয়ে interleaved sequence-এর উপর একটি causal Transformer train করুন। তখন একই weight-গুলো understanding (image -> text) এবং generation (text -> image) দুটোই করে, আর আমরা দুটোই মাপি।
3. **MODALITY GAP কোথা থেকে আসে।** কোনো training-এর আগেই, প্রতিটি randomly initialized encoder যা দেখে সবকিছুকে একটি সরু cone-এ চেপে ধরে, আর ভিন্ন encoder-এর cone ভিন্ন দিকে তাক করা থাকে — তাই দুটি modality sphere-এর আলাদা অঞ্চলে শুরু করে এমন কারণে যার সঙ্গে তাদের বিষয়বস্তুর কোনো সম্পর্ক নেই। আমরা initialization-এ এবং training-এর পরে আবার সেই gap মাপি।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান। প্রতিটি সেকশনের code cell নিজেই তার পরীক্ষা চালায় ও ফলাফল print করে; §3 §1-এ train করা encoder-গুলো (`untrained`, `anchored`, `all_pairs`) পুনরায় ব্যবহার করে, তাই §1 আগে চালাতে হবে।

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

LATENT = 10
EMBED = 16
RAW = {"text": 20, "image": 32, "audio": 28, "depth": 24}

## 1. Emergent alignment: সবকিছু টেক্সটের সঙ্গে বাঁধুন, বাকিটা বিনামূল্যে

চারটি modality (text, image, audio, depth) একটি shared latent concept-এর ভিন্ন noisy rendering। প্রতিটি modality-র জন্য একটি ছোট MLP `Encoder`, আর একটি symmetric InfoNCE loss। `train_bindings` কেবল প্রদত্ত modality জোড়াগুলোর উপর train করে; `retrieval` 128-way gallery-তে R@1 মাপে। চারটি run তুলনা করা হয়: untrained, কেবল text-image, কেবল text-X জোড়া (ImageBind সেটআপ), এবং সব জোড়া (upper bound)। মূল প্রশ্ন: text-X জোড়ায় train করা মডেল কি কখনো না দেখা image->audio জোড়াও retrieve করতে পারে?

In [ ]:
# ===========================================================================
# 1. Emergent alignment: সবকিছু টেক্সটের সঙ্গে বাঁধুন, বাকিটা বিনামূল্যে পান
# ===========================================================================

renders = {m: torch.randn(LATENT, d) for m, d in RAW.items()}


def sample_pair(n, mod_a, mod_b):
    """একই অন্তর্নিহিত concept-এর দুটি view, দুটি ভিন্ন modality-তে।"""
    z = torch.randn(n, LATENT)
    a = z @ renders[mod_a] + 0.6 * torch.randn(n, RAW[mod_a])
    b = z @ renders[mod_b] + 0.6 * torch.randn(n, RAW[mod_b])
    return a, b


class Encoder(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 64), nn.GELU(),
                                 nn.Linear(64, 64), nn.GELU(), nn.Linear(64, EMBED))

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


def infonce(a, b, scale):
    logits = scale.exp() * a @ b.t()
    labels = torch.arange(a.shape[0])
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels))


def train_bindings(pairs, steps=1500, bs=128, lr=3e-3):
    """`pairs` হলো সেই modality জোড়াগুলোর তালিকা যাদের জন্য আমাদের কাছে paired ডেটা আছে।"""
    torch.manual_seed(1)
    enc = nn.ModuleDict({m: Encoder(d) for m, d in RAW.items()})
    scale = nn.Parameter(torch.tensor(math.log(10.0)))
    opt = torch.optim.Adam(list(enc.parameters()) + [scale], lr=lr)
    for step in range(steps):
        ma, mb = pairs[step % len(pairs)]
        xa, xb = sample_pair(bs, ma, mb)
        loss = infonce(enc[ma](xa), enc[mb](xb), scale)
        opt.zero_grad()
        loss.backward()
        opt.step()
        with torch.no_grad():
            scale.clamp_(0, math.log(100.0))
    return enc


@torch.no_grad()
def retrieval(enc, mod_a, mod_b, gallery=128, trials=16):
    hits = 0
    for _ in range(trials):
        xa, xb = sample_pair(gallery, mod_a, mod_b)
        sims = enc[mod_a](xa) @ enc[mod_b](xb).t()
        hits += (sims.argmax(1) == torch.arange(gallery)).sum().item()
    return hits / (gallery * trials)


print("=" * 78)
print("1. EMERGENT CROSS-MODAL ALIGNMENT: bind to text, get the rest free")
print("=" * 78)
print("Four modalities. Retrieval R@1 in 128-way galleries; chance = 0.8%.")
print("Every run trains the SAME number of steps.")
print()

TEXT_ANCHORED = [("text", "image"), ("text", "audio"), ("text", "depth")]
ALL_PAIRS = TEXT_ANCHORED + [("image", "audio"), ("image", "depth"), ("audio", "depth")]

untrained = train_bindings(TEXT_ANCHORED, steps=0)
anchored = train_bindings(TEXT_ANCHORED)
all_pairs = train_bindings(ALL_PAIRS)
image_only = train_bindings([("text", "image")])

rows = [
    ("untrained encoders", untrained),
    ("trained: text-image only", image_only),
    ("trained: text-X pairs only", anchored),
    ("trained: ALL pairs (upper bound)", all_pairs),
]
print(f"{'training data':<34} {'text->image':>12} {'text->audio':>12} {'image->audio':>13} {'audio->depth':>13}")
for name, enc in rows:
    print(f"{name:<34} {retrieval(enc, 'text', 'image'):>11.1%}"
          f" {retrieval(enc, 'text', 'audio'):>11.1%}"
          f" {retrieval(enc, 'image', 'audio'):>12.1%}"
          f" {retrieval(enc, 'audio', 'depth'):>12.1%}")
print()
print("The third row is the ImageBind result. No (image, audio) pair was ever")
print("shown to it, and no (audio, depth) pair either, yet both retrieve far")
print("above chance -- because each modality was pulled toward the same text")
print("embeddings, and being close to the same thing makes them close to each")
print("other. Training every pair explicitly (row 4) is better, as it should be,")
print("but it needs O(n^2) paired datasets instead of O(n).")
print()
print("Row 2 is the control: an encoder that was never bound to the anchor at all")
print("stays at chance for anything involving it. Alignment is not magic -- it")
print("propagates through the anchor, and only to modalities that were bound to")
print("it. That is exactly why text is the anchor in practice: paired")
print("(text, anything) data exists on the web, while (audio, depth) does not.")
print()

## 2. একটি vocabulary, একটি objective: understanding এবং generation

একটি ক্ষুদ্র Chameleon। প্রতিটি concept-এর একটি canonical image-token pattern (16-entry codebook থেকে 4টি token) আর একটি canonical নাম (2টি text token) আছে। Image code, text word এবং দুটি control token (`BOI`, `BOT`) — সব একটি shared vocabulary-তে। `make_sequences` দুই ক্রমে sequence বানায়: caption (`[BOI] img [BOT] txt`, understanding) ও generate (`[BOT] txt [BOI] img`, generation)। `AnyToAny` একটি সাধারণ decoder-only Transformer — কোনো vision tower, projector বা image decoder নেই — যা কেবল next-token cross-entropy দিয়ে train হয়। এরপর মাপা হয় understanding accuracy এবং greedy image generation-এর per-token ও exact-match accuracy, কেবল caption ক্রমে train করা মডেল বনাম দুই ক্রমে train করা মডেলের জন্য।

In [ ]:
# ===========================================================================
# 2. একটি vocabulary, একটি objective: understanding এবং generation
# ===========================================================================

N_CONCEPTS = 12
IMG_CODEBOOK = 16          # discrete "image" vocabulary-র আকার
IMG_TOKENS = 4             # প্রতি ইমেজে token সংখ্যা
TXT_TOKENS = 2             # প্রতি বর্ণনায় token সংখ্যা

# প্রতিটি concept-এর একটি canonical image-token pattern এবং একটি canonical নাম আছে।
concept_img = torch.stack([torch.randint(0, IMG_CODEBOOK, (IMG_TOKENS,))
                           for _ in range(N_CONCEPTS)])
concept_txt = torch.stack([torch.randint(0, N_CONCEPTS, (TXT_TOKENS,))
                           for _ in range(N_CONCEPTS)])

# একটি shared vocabulary: প্রথমে image code, তারপর text word, তারপর দুটি control token।
IMG0 = 0
TXT0 = IMG0 + IMG_CODEBOOK
BOI = TXT0 + N_CONCEPTS            # "begin image"
BOT = BOI + 1                      # "begin text"
VOCAB = BOT + 1
D_MODEL = 96
SEQ = 1 + IMG_TOKENS + 1 + TXT_TOKENS


def make_sequences(b, order):
    """order='caption'  ->  [BOI] img tokens [BOT] text tokens   (understanding)
       order='generate' ->  [BOT] text tokens [BOI] img tokens    (generation)"""
    c = torch.randint(0, N_CONCEPTS, (b,))
    img = concept_img[c].clone()
    # বাস্তবসম্মত noise: প্রতি পাঁচটি image token-এর একটি corrupted, তাই মডেলকে
    # একটি হুবহু pattern মুখস্থ না করে generalize করতে হয়।
    mask = torch.rand(b, IMG_TOKENS) < 0.2
    img[mask] = torch.randint(0, IMG_CODEBOOK, (int(mask.sum()),))
    img = img + IMG0
    txt = concept_txt[c] + TXT0
    boi = torch.full((b, 1), BOI)
    bot = torch.full((b, 1), BOT)
    if order == "caption":
        return torch.cat([boi, img, bot, txt], dim=1), c
    return torch.cat([bot, txt, boi, img], dim=1), c


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(D_MODEL)
        self.attn = nn.MultiheadAttention(D_MODEL, 4, batch_first=True)
        self.ln2 = nn.LayerNorm(D_MODEL)
        self.ff = nn.Sequential(nn.Linear(D_MODEL, 4 * D_MODEL), nn.GELU(),
                                nn.Linear(4 * D_MODEL, D_MODEL))

    def forward(self, x, mask):
        h = self.ln1(x)
        x = x + self.attn(h, h, h, attn_mask=mask, need_weights=False)[0]
        return x + self.ff(self.ln2(x))


class AnyToAny(nn.Module):
    """একটিমাত্র mixed-modal vocabulary-র উপর একটি সাধারণ decoder-only
    Transformer। কোনো vision tower নেই, কোনো projector নেই, এবং কোনো image
    decoder নেই -- একটি ইমেজ অন্য যেকোনো কিছুর মতোই vocabulary entry-র একটি sequence।"""

    def __init__(self, n_layers=3):
        super().__init__()
        self.tok = nn.Embedding(VOCAB, D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, SEQ, D_MODEL) * 0.02)
        self.blocks = nn.ModuleList([Block() for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, VOCAB)

    def forward(self, seq):
        x = self.tok(seq) + self.pos[:, : seq.shape[1]]
        mask = torch.triu(torch.full((seq.shape[1],) * 2, float("-inf")), diagonal=1)
        for blk in self.blocks:
            x = blk(x, mask)
        return self.head(self.ln_f(x))


def train_any(orders, steps=1200, bs=128, lr=2e-3):
    torch.manual_seed(1)
    model = AnyToAny()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(steps):
        order = orders[step % len(orders)]
        seq, _ = make_sequences(bs, order)
        logits = model(seq)
        # পুরো sequence-এর উপর সাধারণ next-token prediction -- Phase 01-এর সেই
        # একই loss, কেবল image token-গুলো vocabulary-তে উপস্থিত।
        loss = F.cross_entropy(logits[:, :-1].reshape(-1, VOCAB), seq[:, 1:].reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model


@torch.no_grad()
def eval_understanding(model, n=2000):
    """Image token ভেতরে, text token বাইরে: এটি কি concept-টির নাম বলতে পারে?"""
    seq, c = make_sequences(n, "caption")
    logits = model(seq)
    pred = logits[:, -(TXT_TOKENS + 1):-1].argmax(-1)         # text-এর position-গুলো
    return (pred == concept_txt[c] + TXT0).all(1).float().mean().item()


@torch.no_grad()
def eval_generation(model, n=2000):
    """Text token ভেতরে, image token বাইরে: greedily generate করুন, তারপর যাচাই
    করুন generate করা token-গুলো concept-টির canonical ইমেজের সঙ্গে মেলে কি না।"""
    seq, c = make_sequences(n, "generate")
    prompt = seq[:, : 1 + TXT_TOKENS + 1]                     # [BOT] text [BOI]
    out = prompt
    for _ in range(IMG_TOKENS):
        nxt = model(out)[:, -1:].argmax(-1)
        out = torch.cat([out, nxt], dim=1)
    gen = out[:, -IMG_TOKENS:] - IMG0
    target = concept_img[c]
    per_token = (gen == target).float().mean().item()
    exact = (gen == target).all(1).float().mean().item()
    return per_token, exact


print("=" * 78)
print("2. ONE VOCABULARY, ONE OBJECTIVE: understanding AND generation")
print("=" * 78)
print("Images are quantized to 4 discrete tokens drawn from a 16-entry codebook")
print("and placed in the same vocabulary as text. One decoder-only Transformer,")
print("plain next-token prediction, trained on both orderings.")
print()

both = train_any(["caption", "generate"])
caption_only = train_any(["caption"])

print(f"{'trained on':<28} {'understanding':>14} {'generation (per-token)':>24} {'exact image':>13}")
for name, m in [("captioning order only", caption_only), ("both orders", both)]:
    u = eval_understanding(m)
    g, e = eval_generation(m)
    print(f"{name:<28} {u:>13.1%} {g:>23.1%} {e:>12.1%}")
print(f"{'chance':<28} {1 / N_CONCEPTS ** TXT_TOKENS:>13.1%}"
      f" {1 / IMG_CODEBOOK:>23.1%} {1 / IMG_CODEBOOK ** IMG_TOKENS:>12.1%}")
print()
print("The same weights that describe an image can draw one, and nothing in the")
print("architecture or the loss distinguishes the two directions -- 'generate an")
print("image' is 'predict the next token' where the next tokens happen to be")
print("image codes. That is the whole idea behind Chameleon-style early-fusion")
print("models, and behind image generation in a chat model.")
print()
print("The first row is the cost of leaving a direction out of the training mix:")
print("a model trained only to caption cannot generate, exactly as in Lesson 6 --")
print("capability follows the data, and 'it has all the parts' is not enough.")
print()
print("Two honest caveats this toy hides. Real image tokenizers (VQ-VAE,")
print("VQ-GAN) lose real detail in quantization, which is why token-based")
print("generation has historically trailed diffusion on image quality; and")
print("mixed-modal training at scale is unstable in ways this small model never")
print("encounters (Chameleon's paper is largely about the norm-growth and")
print("divergence fixes it needed).")
print()

## 3. Modality gap কোথা থেকে আসে

`geometry` শুধু retrieval-এর ক্রম নয়, প্রতিটি modality-র embedding *কোথায়* বসে তা মাপে: একই concept-এর দুটি modality-র মধ্যে cosine ("cos matched"), একই modality-র ভেতরে দুটি random concept-এর cosine ("cos within" — embedding পুরো sphere ব্যবহার করলে ~0, একটি cone-এ ঠাসা থাকলে বড়), এবং দুটি modality centroid-এর দূরত্ব। §1-এর untrained, text-anchored ও all-pairs encoder-গুলোর উপর চালানো হয়, যাতে দেখা যায় gap initialization-এই আছে এবং training এটিকে কতটা বিলীন করে।

In [ ]:
# ===========================================================================
# 3. Modality gap কোথা থেকে আসে
# ===========================================================================

@torch.no_grad()
def geometry(enc, mod_a, mod_b, n=2000):
    """প্রতিটি modality-র embedding কোথায় বসে তা মাপুন, কেবল তাদের ক্রম নয়।"""
    xa, xb = sample_pair(n, mod_a, mod_b)
    ea, eb = enc[mod_a](xa), enc[mod_b](xb)
    matched = (ea * eb).sum(-1).mean().item()          # cosine: একই concept, modality জুড়ে
    within_a = (ea @ ea.t()).mean().item()             # cosine: random জোড়া, একই modality
    within_b = (eb @ eb.t()).mean().item()
    centroid_dist = (ea.mean(0) - eb.mean(0)).norm().item()
    return matched, within_a, within_b, centroid_dist


print("=" * 78)
print("3. WHERE THE MODALITY GAP COMES FROM")
print("=" * 78)
print("'cos matched' compares the same concept across two modalities. 'cos within'")
print("compares two RANDOM concepts inside one modality -- it should be ~0 if the")
print("embeddings use the whole sphere, and large if they are crammed into a cone.")
print()
print(f"{'encoders':<20} {'pair':<15} {'cos matched':>12} {'cos within':>11} {'centroid dist':>14}")
for label, enc in [("UNTRAINED", untrained), ("text-anchored", anchored), ("all pairs", all_pairs)]:
    for a, b in [("text", "image"), ("image", "audio")]:
        mt, wa, wb, cd = geometry(enc, a, b)
        print(f"{label:<20} {a + '-' + b:<15} {mt:>+12.3f} {(wa + wb) / 2:>+11.3f} {cd:>14.3f}")
print()
print("Read the UNTRAINED rows first. Two randomly chosen concepts inside one")
print("modality have cosine ~0.5 -- these encoders map everything into a narrow")
print("cone -- while the SAME concept across two modalities has cosine ~0, and the")
print("two modality centroids sit almost a full unit apart. None of that is about")
print("content. It is the geometry a random deep network starts with: each")
print("encoder has its own cone, and the cones point in different directions.")
print()
print("That is the origin of the 'modality gap' reported for CLIP-style models.")
print("The gap is not created by contrastive learning; it is what contrastive")
print("learning inherits and then has to work against. In the trained rows above")
print("the cones have dissolved (within-modality cosine falls to ~0.01, matched")
print("pairs rise above 0.9) because this toy task is fully learnable with enough")
print("steps. On real data, where alignment is never that complete and the")
print("temperature is clamped, a measurable gap survives training -- which is why")
print("the effect has a name.")
print()
print("Practical consequences worth carrying:")
print()
print("  - absolute cross-modal similarity scores are not comparable to")
print("    within-modality ones, so a cosine threshold tuned on one is wrong on")
print("    the other;")
print("  - arithmetic that mixes modality embeddings (averaging an image and a")
print("    text vector) can land in neither region and behave oddly;")
print("  - and it is one reason a VLM trains a projector (Lesson 3) instead of")
print("    feeding CLIP embeddings straight into an LLM: 'aligned' never meant")
print("    'in the same place'.")